# SETUP

In [31]:
# import the functions from collector.py
import sys
import os
os.chdir("/Users/jakoberhard/Library/CloudStorage/GoogleDrive-jakanterh@gmail.com/My Drive/uni/python/TBA_project")

from src.bjarne_api.collector_new import Fetcher
import pandas as pd
from src.jakob_analysis.functions import filter_train_type, get_possible_destinations, get_connections, create_features

# API CALL

In [38]:
### DEFINE JOURNEY
input = "Berlin Hbf" 

### GET INFOS ON ALL RIDES
# Ausführung im Hauptprogramm 
if __name__ == "__main__":
    fetcher = Fetcher()
    station_name = input
    
    print(f"Suche Station ID für {station_name}...")
    station_id = fetcher.get_station_id(station_name)
    
    if station_id:
        print(f"Station ID gefunden: {station_id}. Suche Fernverkehrsabfahrten (nächste 60min)...")
        departures = fetcher.stations_details(station_id)
        
        if departures:
            print(f"{len(departures)} Abfahrten gefunden. Lade Details...")
            
            all_trips_dfs = []
            
            
            for i, departure in enumerate(departures, start=1):
                if "tripId" in departure:
                    trip_id = departure["tripId"]
                    line_name = departure.get("line", {}).get("name", "Unbekannt")
                    
                    print(f"Lade Trip {i}/{len(departures)}: {line_name}")
                    
                    trip_data = fetcher.trip_details(trip_id)
                    if trip_data is not None:
                        df_trip = fetcher.create_dataframe(trip_data, ride_id=i)
                    
                    if not df_trip.empty:
                        all_trips_dfs.append(df_trip)
                        
                    else: 
                        print(f"Details für Trip {line_name} konnten nicht verarbeitete werden")
            
            # Zusammenfügen aller DataFrames
            if all_trips_dfs:
                final_df = pd.concat(all_trips_dfs, ignore_index=True)
                print("\n--- FERTIG ---")
                print(final_df)
                print(f"\nGesamtanzahl Haltepunkte analysiert: {len(final_df)}")
                print(f"Anzahl Züge: {final_df['ride_id'].nunique()}")
            else:
                print("Konnte keine Trip-Details verarbeiten.")
        else:
            print("Keine passenden Abfahrten im Zeitraum gefunden.")
    else:
        print("Station nicht gefunden.")




Suche Station ID für Berlin Hbf...
Station ID gefunden: 8011160. Suche Fernverkehrsabfahrten (nächste 60min)...
14 Abfahrten gefunden. Lade Details...
Lade Trip 1/14: ICE 697
Lade Trip 2/14: RJ 179
Lade Trip 3/14: ICE 1605
Lade Trip 4/14: IC 2172
Lade Trip 5/14: ICE 504
Lade Trip 6/14: ICE 842
Lade Trip 7/14: ICE 805
Lade Trip 8/14: FLX 1247
Lade Trip 9/14: BUS 40431
Lade Trip 10/14: EN 40457
Lade Trip 11/14: NJ 457
Lade Trip 12/14: IC 2275
Lade Trip 13/14: ICE 607
Lade Trip 14/14: ICE 679

--- FERTIG ---
     ride_id train_name train_type         station_start         station_dest  \
0          1    ICE 697        ICE  Berlin Gesundbrunnen   Frankfurt(Main)Hbf   
1          1    ICE 697        ICE  Berlin Gesundbrunnen   Frankfurt(Main)Hbf   
2          1    ICE 697        ICE  Berlin Gesundbrunnen   Frankfurt(Main)Hbf   
3          1    ICE 697        ICE  Berlin Gesundbrunnen   Frankfurt(Main)Hbf   
4          1    ICE 697        ICE  Berlin Gesundbrunnen   Frankfurt(Main)Hbf   
.. 

# FILTER AND SELECT DATA

In [39]:
# filter data
df_filtered = filter_train_type(final_df)

# get valid destinations (list format)
ls_destinations = get_possible_destinations(df_filtered, input)

# select destination
destination = "Göttingen" # COMES FROM INPUT IN STREAMLIT (USER SHOULD CHOOSE)

# get rides between stations
df_trip = get_connections(df_filtered, input, destination)
df_trip = df_trip.drop(columns = ["current_delay", "stops_total", "stop_index", "stops_remaining"])

df_trip = df_trip.rename({"precip": "precipitation", "temp": "temperature"}, axis=1)

# CREATE FEATURES

In [34]:
from io import BytesIO
hist_delay_station_lookup = pd.read_parquet("data/hist_delay_station_lookup.parquet")
hist_delay_train_lookup = pd.read_parquet("data/hist_delay_train_lookup.parquet")

historical_features_list = [hist_delay_station_lookup, hist_delay_train_lookup]

df_features = create_features(df_trip, api = True, historical_features = historical_features_list)
df_final = df_features[df_features["station_current"] == destination]

In [35]:
def choose_features_target(df):

    cols_exclude = ["ride_id", # id
                    "delay", # target
                    "departure_real","arrival_real", "departure_planned", "arrival_planned", # raw time variables
                    "train_name", "station_current", "station_start", "station_dest", # high-dimensional categories 
                    "hist_delay_train_q90", "hist_delay_station_q90", "stops_total", "stop_index", "share_ride_time" # discarded after correlation analysis
                    ]
    
    feature_cols = [col for col in df.columns if col not in cols_exclude]
    X = df[feature_cols]

    return X

# separate features and target
X = choose_features_target(df_final)

## APPLY MODELS

In [17]:
import joblib

pipeline_hgb_mean = joblib.load("src/jakob_analysis/pipeline_hgb_mean.pkl")
pipeline_hgb_q05 = joblib.load("src/jakob_analysis/pipeline_hgb_q05.pkl")
pipeline_hgb_q95 = joblib.load("src/jakob_analysis/pipeline_hgb_95.pkl")

In [36]:

hgb_step = pipeline_hgb_mean.named_steps['histgradientboostingregressor']

# create dictionary 
model_params = {
    "learning_rate": hgb_step.learning_rate,
    "max_iter": hgb_step.max_iter,
    "max_leaf_nodes": hgb_step.max_leaf_nodes,
    "min_samples_leaf": hgb_step.min_samples_leaf,
}

print(model_params)

{'learning_rate': np.float64(0.010489929587681027), 'max_iter': 485, 'max_leaf_nodes': 34, 'min_samples_leaf': 79}


In [37]:
print(pipeline_hgb_mean.predict(X))
print(pipeline_hgb_q05.predict(X))
print(pipeline_hgb_q95.predict(X))

[ 3.25460338  7.23712494 10.41078359  8.29705256  4.40101227]
[0. 0. 0. 0. 0.]
[12.26253038 24.62064749 31.23933987 27.87588202 15.61974089]
